# Multi-Class Vessel Segmentation on RAVIR Dataset using Implicit Neural Representations

This notebook implements a **SIREN-based Implicit Neural Representation (INR)** model for
3-class retinal vessel segmentation on the [RAVIR dataset](https://ravir.grand-challenge.org/).

## Task
Segment infrared (IR) retinal angiography images into three classes:
- **Class 0** — Background
- **Class 1** — Veins (original mask value: 128)
- **Class 2** — Arteries (original mask value: 255)

## Architecture
The model learns a continuous mapping:

$$f: (x, y, I) \rightarrow \mathbb{R}^3 \quad \text{(3-class probability distribution)}$$

## Key Differences from FIVES (Binary) Pipeline
- **3-class output** with softmax activation instead of sigmoid
- **Dynamic patching** with reflection padding via a `Patcher` class
- **Multi-class Focal-Dice loss** with per-class Dice computation
- **Longer training** (800 epochs) due to harder multi-class objective
- **Custom collate function** for variable-length batches

## 1. Imports and Configuration

In [ ]:
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset

# ── Configuration ─────────────────────────────────────────────────────────────
IMAGE_SIZE = 256              # Target image size after resizing
PATCH_SIZE = (258, 258)       # Patcher patch dimensions (with reflection padding)
BATCH_SIZE = 64               # Images per training batch
NUM_EPOCHS = 800              # Training epochs (multi-class requires longer training)
LEARNING_RATE = 1e-3          # Initial learning rate for Adam
NUM_CLASSES = 3               # Background, Vein, Artery
NUM_LAYERS = 4                # SIREN MLP depth
HIDDEN_DIM = IMAGE_SIZE       # Hidden layer width
NUM_FREQS = 5                 # Positional encoding frequency bands
ALPHA = 0.8                   # Focal loss alpha
GAMMA = 0.5                   # Focal loss gamma
SMOOTH = 1e-5                 # Dice loss smoothing constant
DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Using device: {DEVICE}')

## 2. Patcher — Dynamic Patch Extraction with Reflection Padding

The `Patcher` class handles splitting images into non-overlapping patches and
reconstructing them. When image dimensions are not evenly divisible by the patch
size, reflection padding is applied to avoid losing information at the borders.

This is more sophisticated than the fixed grid approach used for FIVES, because
RAVIR images can be of varying sizes.

In [ ]:
class Patcher:
    """Utility for splitting images into patches and reassembling them.

    Uses torch.nn.functional.unfold/fold for efficient patch extraction.
    Applies reflection padding when image dimensions aren't divisible by
    patch size, preventing information loss at boundaries.

    Args:
        patch_shape: Target patch size as (height, width).
    """

    def __init__(self, patch_shape):
        assert len(patch_shape) == 2, "Only 2D patches are supported."
        self.patch_shape = patch_shape

    def patch(self, data):
        """Split a (C, H, W) tensor into non-overlapping patches.

        Returns:
            patches: (num_patches, C, pH, pW) tensor.
            original_shape: (H, W) tuple for reconstruction.
        """
        assert data.ndim == 3, "Input must be (channels, height, width)."
        channels, height, width = data.shape
        patch_h, patch_w = self.patch_shape

        # Pad image so dimensions are divisible by patch size
        pad_h, pad_w = self._get_padding((height, width))
        padded = F.pad(data, (0, pad_w, 0, pad_h), mode='reflect')

        # Extract patches using unfold
        patches = F.unfold(
            padded.unsqueeze(0),
            kernel_size=self.patch_shape,
            stride=self.patch_shape
        )
        patches = patches.reshape(channels, patch_h, patch_w, -1).permute(3, 0, 1, 2)
        return patches, (height, width)

    def unpatch(self, patches, original_shape):
        """Reconstruct original image from patches.

        Args:
            patches: (num_patches, C, pH, pW) tensor.
            original_shape: (H, W) from the patch() call.

        Returns:
            Reconstructed (C, H, W) tensor.
        """
        height, width = original_shape
        pad_h, pad_w = self._get_padding((height, width))
        padded_shape = (height + pad_h, width + pad_w)

        num_patches, channels, _, _ = patches.shape
        patches = patches.permute(1, 2, 3, 0).reshape(1, -1, num_patches)

        padded_image = F.fold(
            patches, output_size=padded_shape,
            kernel_size=self.patch_shape, stride=self.patch_shape
        )

        # Remove padding to restore original dimensions
        return padded_image[0, :, :height, :width]

    def _get_padding(self, spatial_shape):
        """Compute padding to make dimensions divisible by patch size."""
        h, w = spatial_shape
        ph, pw = self.patch_shape
        pad_h = (ph - h % ph) if h % ph != 0 else 0
        pad_w = (pw - w % pw) if w % pw != 0 else 0
        return pad_h, pad_w

## 3. Dataset

The `SegmentationDataset` for RAVIR:
1. Loads IR angiography images and multi-class masks
2. Maps raw mask values {0, 128, 255} → class indices {0, 1, 2}
3. Splits each image into patches via the `Patcher`
4. Converts each patch to (x, y, intensity) → label pairs for the INR

A **custom collate function** concatenates variable-length samples along the pixel
dimension (since different images may produce different total pixel counts).

In [ ]:
class SegmentationDataset(Dataset):
    """RAVIR dataset for multi-class vessel segmentation (3 classes).

    Mask value mapping:
        0   → Class 0 (background)
        128 → Class 1 (vein)
        255 → Class 2 (artery)

    Args:
        images_path: Directory of IR angiography images.
        masks_path: Directory of segmentation masks.
        target_size: Resize to (H, W). None keeps original resolution.
        patch_size: Patch dimensions for the Patcher.
        augment: Whether to apply data augmentation.
    """

    def __init__(self, images_path, masks_path, target_size=(IMAGE_SIZE, IMAGE_SIZE),
                 patch_size=PATCH_SIZE, augment=True):
        self.images_path = images_path
        self.masks_path = masks_path
        self.image_files = sorted(os.listdir(images_path))
        self.mask_files = sorted(os.listdir(masks_path))
        self.target_size = target_size
        self.patch_size = patch_size
        self.augment = augment

        # Initialize patcher for dynamic patch extraction
        self.patcher = Patcher(self.patch_size)

        # Lighter augmentation for IR images (more sensitive to noise artifacts)
        self.augmentations = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=5, p=0.5),
            A.RandomBrightnessContrast(p=0.1),
            A.GaussianBlur(blur_limit=1, p=0.1),
            A.GaussNoise(var_limit=(5.0, 20.0), p=0.1),
            A.CLAHE(clip_limit=1, p=0.1),
        ], additional_targets={'mask': 'image'})

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        # Load grayscale IR image and multi-class mask
        image_path = os.path.join(self.images_path, self.image_files[idx])
        mask_path = os.path.join(self.masks_path, self.mask_files[idx])
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # Resize if specified
        if self.target_size is not None:
            image = cv2.resize(image, self.target_size, interpolation=cv2.INTER_LINEAR)
            mask = cv2.resize(mask, self.target_size, interpolation=cv2.INTER_NEAREST)

        # Apply augmentation
        if self.augment:
            augmented = self.augmentations(image=image, mask=mask)
            image, mask = augmented['image'], augmented['mask']

        # Normalize image intensities to [0, 1]
        image = image.astype(np.float32) / 255.0

        # Map mask pixel values {0, 128, 255} → class indices {0, 1, 2}
        mask = self._transform_mask(mask).astype(np.float32)

        # Convert to tensors with channel dimension for the Patcher
        image_tensor = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        mask_tensor = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        # Extract patches using reflection padding
        patches_image, _ = self.patcher.patch(image_tensor)
        patches_mask, _ = self.patcher.patch(mask_tensor)

        # Convert each patch to coordinate-intensity pairs
        coords_list, labels_list = [], []
        for img_patch, mask_patch in zip(patches_image, patches_mask):
            ci, lab = self._generate_coords_and_labels(img_patch, mask_patch)
            coords_list.append(ci)
            labels_list.append(lab)

        # Concatenate all patches into a single tensor per image
        coords_intensities = torch.cat(coords_list, dim=0)
        labels = torch.cat(labels_list, dim=0)

        return coords_intensities, labels

    @staticmethod
    def _transform_mask(mask):
        """Map raw pixel values {0, 128, 255} → class indices {0, 1, 2}."""
        transformed = np.zeros_like(mask)
        transformed[mask == 0] = 0    # Background
        transformed[mask == 128] = 1  # Vein
        transformed[mask == 255] = 2  # Artery
        return transformed

    @staticmethod
    def _generate_coords_and_labels(image_patch, mask_patch):
        """Convert a patch pair to (x, y, intensity) features and class labels.

        Args:
            image_patch: (1, H, W) image tensor.
            mask_patch: (1, H, W) mask tensor with class indices.

        Returns:
            coords_intensities: (H*W, 3) tensor.
            labels: (H*W,) tensor of class indices.
        """
        C, H, W = image_patch.shape

        # Create normalized coordinate grid
        y_coords, x_coords = torch.meshgrid(
            torch.arange(H), torch.arange(W), indexing='ij'
        )
        coords = torch.stack((x_coords, y_coords), dim=-1).view(-1, 2).float()
        coords = coords / torch.tensor([W, H], dtype=torch.float32)

        intensities = image_patch.view(-1, 1)
        labels = mask_patch.view(-1)

        coords_intensities = torch.cat([coords, intensities], dim=-1)
        return coords_intensities, labels


def custom_collate(batch):
    """Custom collate: concatenate variable-length samples along pixel dim.

    Required because different images may produce different total pixel counts
    after patching (due to varying image sizes).
    """
    coords_list, labels_list = zip(*batch)
    return torch.cat(coords_list, dim=0), torch.cat(labels_list, dim=0)

### 3.1 Initialize Dataset and DataLoader

In [ ]:
# ── Dataset paths ─────────────────────────────────────────────────────────────
# Using pre-extracted high-variance patches for more efficient training
IMAGES_PATH = 'data/RAVIR/train/patch_images'
MASKS_PATH = 'data/RAVIR/train/patch_masks'

# Initialize dataset without augmentation for development
dataset = SegmentationDataset(
    IMAGES_PATH, MASKS_PATH,
    target_size=None,      # Patches are already 256×256
    augment=False
)
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=custom_collate
)

# Sanity check
sample_coords, sample_labels = next(iter(dataloader))
print(f'Coords+Intensity shape: {sample_coords.shape}')
print(f'Labels shape: {sample_labels.shape}')
print(f'Unique labels: {torch.unique(sample_labels).tolist()}')

### 3.2 Visualize Sample Patches

In [ ]:
def show_augmented_images(dataset, num_images=3):
    """Display sample patches with their masks for visual verification."""
    for i in range(num_images):
        coords_intensities, labels = dataset[i]

        coords = coords_intensities[:, :2]
        intensities = coords_intensities[:, 2]

        H = int(coords[:, 1].max().item() * dataset.patch_size[0]) + 1
        W = int(coords[:, 0].max().item() * dataset.patch_size[1]) + 1

        image = torch.zeros((H, W))
        mask = torch.zeros((H, W))

        x_idx = (coords[:, 0] * (W - 1)).long()
        y_idx = (coords[:, 1] * (H - 1)).long()

        image[y_idx, x_idx] = intensities
        mask[y_idx, x_idx] = labels

        fig, axs = plt.subplots(1, 2, figsize=(8, 4))
        axs[0].imshow(image.numpy(), cmap='gray')
        axs[0].set_title('Patched Image')
        axs[0].axis('off')
        axs[1].imshow(mask.numpy(), cmap='gray')
        axs[1].set_title('Patched Mask')
        axs[1].axis('off')
        plt.tight_layout()
        plt.show()


show_augmented_images(dataset)

## 4. Model Architecture

Same SIREN-based architecture as the binary model, but extended for multi-class:
- Output dimension = 3 (one per class)
- Last layer uses softmax via `SineLayer(is_last=True)`
- Can optionally use a linear output layer for more stable gradients

In [ ]:
class PositionalEncoding(nn.Module):
    """Fourier feature encoding: (x,y) → sin/cos at exponentially increasing frequencies."""

    def __init__(self, num_freqs):
        super().__init__()
        self.num_freqs = num_freqs

    def forward(self, x):
        frequencies = torch.linspace(0, self.num_freqs - 1, self.num_freqs, device=x.device)
        frequencies = 2.0 ** frequencies
        frequencies = frequencies.view(1, 1, -1)

        x = x.unsqueeze(-1)
        x = x * frequencies
        x = torch.cat([torch.sin(x), torch.cos(x)], dim=-1)
        x = x.view(x.shape[0], -1)
        return x


class AdaptiveDropout(nn.Module):
    """Dropout with exponentially decaying probability."""

    def __init__(self, initial_p=0.5, decay_factor=0.95):
        super().__init__()
        self.p = initial_p
        self.decay_factor = decay_factor

    def forward(self, x):
        return F.dropout(x, p=self.p, training=True) if self.training else x

    def step(self):
        self.p *= self.decay_factor


class SineLayer(nn.Module):
    """SIREN layer: sin(ω₀ · (Wx + b)) with optional softmax for output layers.

    Args:
        is_last: If True, applies softmax after sine (for multi-class output).
    """

    def __init__(self, in_features, out_features, bias=True, is_first=False,
                 omega_0=1.0, is_last=False):
        super().__init__()
        self.omega_0 = omega_0
        self.is_first = is_first
        self.is_last = is_last
        self.in_features = in_features
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        self._init_weights()

    def _init_weights(self):
        with torch.no_grad():
            if self.is_first:
                bound = 1.0 / self.in_features
            else:
                bound = np.sqrt(6.0 / self.in_features) / self.omega_0
            self.linear.weight.uniform_(-bound, bound)

    def forward(self, x):
        x = torch.sin(self.omega_0 * self.linear(x))
        if self.is_last:
            x = F.softmax(x, dim=-1)
        return x


class INRSegmentationModel(nn.Module):
    """Multi-class INR segmentation model.

    Input:  (N, 3) — [x_norm, y_norm, intensity]
    Output: (N, num_classes) — class probabilities
    """

    def __init__(self, num_classes, hidden_dim=256, num_layers=5, num_freqs=10,
                 initial_dropout_p=0.5, outermost_linear=False, linear_network=False):
        super().__init__()
        self.num_classes = num_classes
        self.outermost_linear = outermost_linear

        self.pos_enc = PositionalEncoding(num_freqs)
        num_coords = 2
        input_dim = num_coords * num_freqs * 2 + 1  # +1 for intensity

        self.reduction_layer = nn.Linear(input_dim, hidden_dim)

        # Build MLP
        self.mlp = nn.ModuleList()

        # First layer with special initialization
        self.mlp.append(nn.Sequential(
            SineLayer(hidden_dim, hidden_dim, is_first=True),
            nn.BatchNorm1d(hidden_dim),
        ))

        # Intermediate layers
        for _ in range(1, num_layers - 2):
            if linear_network:
                self.mlp.append(nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim),
                    nn.ReLU(),
                    nn.BatchNorm1d(hidden_dim),
                ))
            else:
                self.mlp.append(nn.Sequential(
                    SineLayer(hidden_dim, hidden_dim),
                    nn.BatchNorm1d(hidden_dim),
                ))

        # Output layer
        if outermost_linear:
            self.mlp.append(nn.Linear(hidden_dim, num_classes))
            self.softmax = nn.Softmax(dim=1)
        else:
            self.mlp.append(SineLayer(hidden_dim, num_classes, is_last=True))

    def forward(self, coords_intensities):
        coords = coords_intensities[:, :-1]      # (N, 2)
        intensities = coords_intensities[:, -1:]  # (N, 1)

        x = self.pos_enc(coords)                  # (N, encoded_dim)
        x = torch.cat([x, intensities], dim=-1)   # (N, input_dim)
        x = self.reduction_layer(x)               # (N, hidden_dim)

        for layer in self.mlp[:-1]:
            x = layer(x)

        x = self.mlp[-1](x)                       # (N, num_classes)

        if self.outermost_linear:
            x = self.softmax(x)

        return x

## 5. Loss Function — Multi-Class Focal-Dice

Extension of the binary Focal-Dice loss to K classes:
- Uses one-hot encoding for multi-class targets
- Dice loss is computed **per class** then averaged (class-balanced)
- This ensures the model doesn't ignore minority classes (vessels)

In [ ]:
class FocalDiceLoss(nn.Module):
    """Combined Focal + per-class Dice loss for multi-class segmentation.

    Total Loss = Focal_Loss + (1 - mean_Dice)

    Args:
        num_classes: Number of segmentation classes.
        alpha: Focal loss weight for ground truth class.
        gamma: Focal loss focusing parameter.
        smooth: Dice smoothing constant.
    """

    def __init__(self, num_classes, alpha=0.8, gamma=2, smooth=1e-5):
        super().__init__()
        self.num_classes = num_classes
        self.alpha = alpha
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, inputs, targets):
        targets = targets.long()

        # Convert class indices to one-hot encoding: (N,) → (N, K)
        targets_one_hot = torch.zeros_like(inputs).scatter_(1, targets.unsqueeze(1), 1)

        # ── Multi-class Focal Loss ────────────────────────────────────────────
        focal_loss = -self.alpha * (1 - inputs) ** self.gamma * targets_one_hot * torch.log(inputs + 1e-8)
        focal_loss = focal_loss.sum(dim=1).mean()

        # ── Per-class Dice Loss ───────────────────────────────────────────────
        intersection = (inputs * targets_one_hot).sum(dim=0)  # (K,)
        union = inputs.sum(dim=0) + targets_one_hot.sum(dim=0)  # (K,)
        dice_per_class = (2.0 * intersection + self.smooth) / (union + self.smooth)
        dice_loss = 1 - dice_per_class.mean()

        return focal_loss + dice_loss

## 6. Hyperparameters

In [ ]:
print(f'Number of classes:       {NUM_CLASSES}')
print(f'Number of layers:        {NUM_LAYERS}')
print(f'Hidden dimension:        {HIDDEN_DIM}')
print(f'Positional encoding L:   {NUM_FREQS}')
print(f'Focal loss alpha:        {ALPHA}')
print(f'Focal loss gamma:        {GAMMA}')
print(f'Batch size:              {BATCH_SIZE}')
print(f'Learning rate:           {LEARNING_RATE}')
print(f'Number of epochs:        {NUM_EPOCHS}')

## 7. Training Loop

In [ ]:
# ── Model, optimizer, loss, scheduler ─────────────────────────────────────────
model = INRSegmentationModel(
    num_classes=NUM_CLASSES,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    outermost_linear=False,
    num_freqs=NUM_FREQS,
    linear_network=False,
).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = FocalDiceLoss(num_classes=NUM_CLASSES, alpha=ALPHA, gamma=GAMMA, smooth=SMOOTH)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=30, verbose=True)

# ── Training ──────────────────────────────────────────────────────────────────
losses = []
MODEL_SAVE_PATH = 'checkpoints/ravir_multiclass.pth'
os.makedirs('checkpoints', exist_ok=True)

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0

    for coords_intensities, labels in dataloader:
        coords_intensities = coords_intensities.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        # Forward pass: predict class probabilities for each pixel
        outputs = model(coords_intensities)  # (N, 3)
        outputs = outputs.view(-1, NUM_CLASSES)
        labels = labels.view(-1)

        # Multi-class Focal-Dice loss
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= len(dataloader)
    losses.append(epoch_loss)
    scheduler.step(epoch_loss)

    if epoch == 0 or (epoch + 1) % 50 == 0:
        print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {epoch_loss:.4f}')

### 7.1 Training Loss Curve

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, NUM_EPOCHS + 1), losses, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Focal-Dice Loss')
plt.title('Training Loss over Epochs')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7.2 Save Model

In [ ]:
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f'Model saved to {MODEL_SAVE_PATH}')

## 8. Evaluation & Visualization

Color coding for the 3-class segmentation:
- **White** — Background (class 0)
- **Blue** — Vein (class 1)
- **Red** — Artery (class 2)

In [ ]:
def visualize_segmentation_mask(mask, title='Segmentation Mask'):
    """Render a 3-class mask with color coding: white/blue/red."""
    color_mapping = {
        0: [255, 255, 255],  # Background: white
        1: [0, 0, 255],      # Vein: blue
        2: [255, 0, 0],      # Artery: red
    }

    H, W = mask.shape
    rgb = np.zeros((H, W, 3), dtype=np.uint8)
    for class_val, color in color_mapping.items():
        rgb[mask == class_val] = color

    plt.imshow(rgb / 255.0)
    plt.axis('off')
    plt.title(title)


def generate_coords_intensities(image_patch, device):
    """Generate (x, y, intensity) vectors from a single patch for inference."""
    C, H, W = image_patch.shape

    y_coords, x_coords = torch.meshgrid(
        torch.arange(H, device=device),
        torch.arange(W, device=device), indexing='ij'
    )
    coords = torch.stack((x_coords, y_coords), dim=-1).view(-1, 2).float()
    coords = coords / torch.tensor([W - 1, H - 1], dtype=torch.float32, device=device)

    intensities = image_patch.view(-1).to(device).unsqueeze(-1)
    return torch.cat([coords, intensities], dim=-1)


def evaluate_model(model, image, img_size, save_path=None, image_name=None):
    """Run multi-class inference on a test image and visualize results.

    Pipeline:
    1. Normalize and patch the image
    2. Generate coordinate-intensity features per patch
    3. Run model inference (argmax for class prediction)
    4. Reconstruct full mask from patch predictions
    """
    model.eval()
    patcher = Patcher(PATCH_SIZE)

    # Normalize and convert to tensor
    image_normalized = image.astype(np.float32) / 255.0
    image_tensor = torch.tensor(image_normalized, dtype=torch.float32).unsqueeze(0).to(DEVICE)

    # Split into patches
    patches_image, _ = patcher.patch(image_tensor)

    # Generate features for all patches
    coords_list, patch_sizes = [], []
    for img_patch in patches_image:
        if img_patch.dim() == 2:
            img_patch = img_patch.unsqueeze(0)
        ci = generate_coords_intensities(img_patch, device=DEVICE)
        coords_list.append(ci)
        patch_sizes.append((img_patch.shape[1], img_patch.shape[2]))

    all_coords = torch.cat(coords_list, dim=0).to(DEVICE)

    # Inference
    with torch.no_grad():
        outputs = model(all_coords)
        predicted_labels = outputs.argmax(dim=-1)

    # Reconstruct mask from patches
    start = 0
    predicted_patches = []
    for (H, W) in patch_sizes:
        n_pixels = H * W
        patch_labels = predicted_labels[start:start + n_pixels].view(H, W)
        predicted_patches.append(patch_labels)
        start += n_pixels

    predicted_tensor = torch.stack(predicted_patches).unsqueeze(1).float()
    predicted_mask = patcher.unpatch(predicted_tensor, img_size)
    predicted_mask = predicted_mask.long().cpu().numpy().astype(np.uint8).squeeze()

    # Visualization
    if save_path and image_name:
        output_path = os.path.join(save_path, f'{image_name}_predicted_mask.png')
        cv2.imwrite(output_path, predicted_mask)
        print(f'Saved predicted mask to: {output_path}')
    else:
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(image, cmap='gray')
        plt.title('Original Image')
        plt.axis('off')

        plt.subplot(1, 2, 2)
        visualize_segmentation_mask(predicted_mask, title='Predicted Mask')
        plt.tight_layout()
        plt.show()

    return predicted_mask

In [ ]:
# ── Load model and run evaluation ─────────────────────────────────────────────
model = INRSegmentationModel(
    num_classes=NUM_CLASSES, hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS, outermost_linear=False,
    num_freqs=NUM_FREQS, linear_network=False,
).to(DEVICE)
model.load_state_dict(torch.load(MODEL_SAVE_PATH))

# Test on a sample image
TEST_IMAGE_PATH = 'data/RAVIR/test/IR_Case_006.png'
test_image = cv2.imread(TEST_IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
test_image = cv2.resize(test_image, (256, 256), interpolation=cv2.INTER_LINEAR)

predicted_mask = evaluate_model(model, test_image, (256, 256))

## 9. Experiment Log & Ablation Results

| Layers | Outer Linear | α | γ | Freq | Batch | Loss | Notes |
|--------|-------------|-----|-----|------|-------|------|-------|
| 6 | True | 0.8 | 2 | 10 | 16 | 1.2482 | LR=1e-3 works better than 1e-2 |
| 3 | True | 0.8 | 2 | 10 | 64 | 1.3324 | Larger batch slight improvement |
| 6 | False | 0.8 | 2 | 10 | 16 | 1.1937 | Softmax output |
| 9 | False | 0.8 | 2 | 10 | 16 | 0.9873 | More layers → lower loss |
| 9 | False | 0.8 | 2 | 10 | 16 | 0.91 | With dropout → predicts all white |

**Key findings:**
1. Learning rate 1e-3 is more stable than 1e-2
2. Softmax output (non-linear) outperforms linear output layer
3. More layers improve loss but risk degenerate predictions with dropout
4. Cross-entropy vs. Focal-Dice: Focal-Dice handles class imbalance better

### Next Steps
1. Experiment with scheduler alternatives to escape local minima
2. Try without normalization to avoid over-smoothing
3. Evaluate on larger external datasets (e.g., [HRF](https://paperswithcode.com/dataset/hrf))